# 深度学习模型数据装载与高级增强管线 (Data Preparation & Augmentation Pipeline)

**职责说明 (Responsibility):**
本模块负责将经过 EDA 与物理清洗后的静态图像数据（`Integrated_Dataset_384`）动态转化为适用于 PyTorch 模型训练的张量流 (Tensor Stream)。

**核心技术点 (Core Techniques):**
1. **基于权重的随机采样 (WeightedRandomSampler):** 解决长尾分布导致的类别极度不平衡问题。
2. **环境扰动模拟 (ColorJitter):** 引入光照与对比度变化，迫使模型在复杂野外背景中聚焦于物理材质的本质特征。
3. **CutMix / MixUp 局部上下文合成技术:** 突破 TACO 等有限背景带来的泛化瓶颈，通过跨类随机融合，极大提升模型面对未知噪声的鲁棒性 (Robustness)。

## 1. 导入依赖库与基础配置 (Dependencies & Configuration)

In [2]:
import torch
from torchvision import datasets
from torchvision.transforms import v2  # 必须使用最新版 v2 接口以支持高级合成增强
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 配置全局参数
DATASET_DIR = "./data/Integrated_Dataset_384"  # 从 PKU Web Disk 下载并解压后的统一数据集路径
BATCH_SIZE = 32
NUM_WORKERS = 4

print("[系统] 数据装载环境初始化成功。准备接入底层数据。")

ImportError: DLL load failed while importing _imaging: 操作系统无法运行 %1。

## 2. 基础预处理管线构建 (Base Transform Pipeline)
在单张图像层面执行基础增强与张量化操作。

In [ ]:
print("\n[系统] 正在构建实时数据流预处理管线...")

# 训练集基础管线 (含几何与色彩扰动)
train_base_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(15), 
    # 核心：应对高熵背景的光照剧烈变化模拟
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2), 
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    # 匹配 ImageNet 预训练权重体系的标准归一化
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集/测试集管线 (严格禁止任何随机增强)
val_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 实例化抽象数据集 (自动依据文件夹名称构建标签)
train_dataset = datasets.ImageFolder(root=DATASET_DIR, transform=train_base_transforms)
print(f"\n[信息] 物理材质分类到张量标签的自动映射结果 (class_to_idx):\n{train_dataset.class_to_idx}")

## 3. 长尾分布权重均衡化采样器配置 (Handling Class Imbalance)
根据 EDA 分析结果，动态计算各分类的倒数权重，强制模型在每个 Epoch 内获得均等数量的少样本类（如 shoes, clothes）反馈。

In [ ]:
targets = train_dataset.targets
class_counts = np.bincount(targets)
print(f"\n[分析] 各类别原始样本绝对数量: {class_counts}")

# 利用频率的倒数构建补偿权重
class_weights = 1.0 / class_counts
sample_weights = class_weights[targets]

# 实例化带有放回重采样的加权采样器
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).double(),
    num_samples=len(sample_weights),
    replacement=True 
)

# 最终装配高性能数据加载器 (注: 启用 sampler 时必须禁用 shuffle)
train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=BATCH_SIZE, 
    sampler=sampler, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)
print("[系统] 应对极度不平衡分布的数据加载器装配完成！")

## 4. CutMix / MixUp 高级上下文合成算子定义 (Advanced Synthesis)
**注意:** 传统的增强是在单张图像上操作，而 CutMix/MixUp 是在 **Batch 层面**混合多张图像及其标签。因此，必须将其定义为独立的算子，并在模型训练循环内部调用。

In [ ]:
NUM_CLASSES = len(train_dataset.classes)

# 定义 CutMix 算子：将随机图像的局部区域裁剪并粘贴至当前图像 (α=1.0 产生较均匀的 Beta 分布采样)
cutmix = v2.CutMix(num_classes=NUM_CLASSES, alpha=1.0)
# 定义 MixUp 算子：以透明度比例直接融合两张图像的像素值 (α=0.2 保留较多主图特征)
mixup = v2.MixUp(num_classes=NUM_CLASSES, alpha=0.2)

# 构建随机选择器：在每个 Batch 中，随机决定执行 CutMix 或 MixUp，以丰富噪声的多样性
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

print("\n🚀 [大功告成] 数据引擎准备就绪！")
print("💡 团队成员须知：在编写 `Model_Training.ipynb` 的训练循环时，请务必执行以下操作：")
print("   1. 使用 `nn.CrossEntropyLoss(label_smoothing=...)` 作为损失函数 (因标签已转化为浮点软标签)。")
print("   2. 在前向传播前注入算子：`images, labels = cutmix_or_mixup(images, labels)`")